In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
import os

In [22]:
# Cargamos los csv de los tifs
path = "saved_files/dataset"
dfs = {}
for archivo in os.listdir(path):
    if archivo.endswith("_features.csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)

In [23]:
dfs_to_keep = [
    "C2X_3x3_merge_depth_lt_1", "C2X_5x5_merge_depth_lt_1", "C2X-Complex_5x5_merge_depth_lt_1", "C2X-Complex_3x3_merge_depth_lt_1",
    "C2X_3x3_merge_depth_gt_1", "C2X_5x5_merge_depth_gt_1", "C2X-Complex_5x5_merge_depth_gt_1", "C2X-Complex_3x3_merge_depth_gt_1",
    #"C2RCC_3x3_merge_depth_lt_1", "C2RCC_5x5_merge_depth_gt_1", "C2RCC_5x5_merge_depth_lt_1", "C2RCC_3x3_merge_depth_gt_1",
]

In [24]:
dfs = {k: dfs[k] for k in dfs_to_keep if k in dfs}

for nombre_df, df in dfs.items():
    for band_set in ["rhow", "rhown"]:
        dfs[nombre_df] = df.dropna()

In [25]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Invierno'
    elif month in [3, 4, 5]:
        return 'Primavera'
    elif month in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'
    

def get_zone(buoy):
    if buoy in ["CTD1", "CTD2", "CTD3", "CTD4"]:
        return 'Zona-1'
    elif buoy in ["CTD6", "CTD8", "CTD9", "CTD10", "CTD12"]:
        return 'Zona-2'
    elif buoy in ["CTD7"]:
        return 'Zona-3'
    elif buoy in ["CTD11"]:
        return 'Zona-4'

In [26]:
for nombre_df, df in dfs.items():
    df["High_Chl"] = df["Chl"]>5
    df['Date'] = pd.to_datetime(df['Date'])
    df['Season'] = df['Date'].dt.month.apply(get_season)
    df['Zone'] = df['Buoy'].apply(get_zone)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category')
    dfs[nombre_df] = df
    

In [7]:
def filtrar_columnas(df):

    # 1. Separar columnas numéricas y no numéricas
    df_numericas = df.select_dtypes(include='number')
    df_no_numericas = df.select_dtypes(exclude='number')

    # 2. Calcular la correlación con 'Chl' solo entre columnas numéricas
    correlaciones = df_numericas.corr()['Chl'].drop('Chl')

    # 3. Filtrar predictores numéricos con correlación significativa
    umbral_corr = 0.1
    columnas_utiles = correlaciones[correlaciones.abs() >= umbral_corr].index.tolist()

    # 4. Reconstruir el DataFrame con:
    # - Las columnas numéricas útiles
    # - La columna objetivo 'Chl'
    df_filtrado = pd.concat([df[columnas_utiles + ['Chl']]], axis=1)

    # Calcular la matriz de correlación entre predictores
    corr_matrix = df_filtrado.drop(columns='Chl').corr().abs()

    # Seleccionar columnas a eliminar (altamente correlacionadas entre sí)
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    columnas_redundantes = [col for col in upper.columns if any(upper[col] > 0.98)]

    # Eliminar redundantes
    df_final = pd.concat([df_filtrado.drop(columns=columnas_redundantes), df_no_numericas], axis=1)
    df_final = df_final.drop(columns=["Date", "Buoy"])

    return df_final


In [8]:
for nombre_df, df in dfs.items():
    dfs[nombre_df] = filtrar_columnas(df)

In [27]:
df = dfs["C2X_3x3_merge_depth_lt_1"]

### Selección de hiperparámetros con Optuna

In [15]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, train_test_split
import optuna
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error
#from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import time
import joblib

In [36]:
df= df.iloc[:,4:]
train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df["High_Chl"])


In [38]:
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

In [31]:
def cross_validation(trial, df, target, model_name):
    
    # Separamos en X e y
    #X = df.loc[:, df.columns != target]
    X = df.drop(columns=[target, 'High_Chl'])
    y = df[target]
    y_class = df["High_Chl"]
    
    # Definimos los folds
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    oof_preds = np.zeros(len(df))  # Almacenar las predicciones OOF

    start = time.time()
    
    if model_name == "LBM":
        params_lbm = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'device': 'cpu',  # usa CPU/GPU
            'verbosity': -1,
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves': trial.suggest_categorical('num_leaves', [20, 40, 60, 80, 100]),
            'max_depth': trial.suggest_int('max_depth', 5, 8),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'n_estimators': trial.suggest_categorical('n_estimators', [500, 1000, 2000]),
            'early_stopping_rounds': 50,
        }

    if model_name == "XGB":
        params_xgb = {
            'n_estimators': trial.suggest_categorical('n_estimators', [500, 1000, 2000]),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 5, 8),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 4),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'device': 'cpu',
            'objective': 'reg:squarederror',
            'tree_method': 'hist',
            'enable_categorical': True,
            'early_stopping_rounds': 50,
            'eval_metric': 'rmse'
        }
    
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
        print(f"Fold {fold+1}")
        
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        if model_name == "XGB":
            model = XGBRegressor(**params_xgb)
            model.fit(X_train, y_train, 
                      eval_set=[(X_val, y_val)], 
                      verbose=0)
            
        if model_name == "LBM":
            model = LGBMRegressor(**params_lbm)
            model.fit(X_train, y_train,
                eval_set=[(X_val, y_val)])
        
        # Guardamos las predicciones en su sitio correspondiente
        oof_preds[val_idx] = model.predict(X_val)

    print(f"Running time: {time.time() - start:.1f} sec")
    # Calculamos el RMSE OOF
    rmse_score = np.sqrt(mean_squared_error(y, oof_preds))
    print(f"OOF RMSE: {rmse_score:.4f}")
    
    return rmse_score

In [54]:
def run_optuna(df, target, n_trials, model_names):
    results = {}

    for model_name in model_names:
        print(f"Buscando mejores hiperparámetros para {model_name}...")
        study = optuna.create_study(direction='minimize')
        study.optimize(lambda trial: cross_validation(trial, df, target, model_name), n_trials=n_trials)
        print(f"\n✅ {model_name} - Mejor RMSLE: {study.best_value:.4f}")
        print(f"📋 Parámetros: {study.best_params}\n")
        
        results[model_name] = {
            'best_params': study.best_params,
            'best_score': study.best_value,
            'study': study
        }
    return results

In [55]:
model_names = ["XGB", "LBM"]
n_trials = 20
results = run_optuna(train, "Chl", n_trials, model_names)

[I 2025-06-24 12:22:47,253] A new study created in memory with name: no-name-fa720214-6f8c-4099-99f4-edf6841d7c5d


Buscando mejores hiperparámetros para XGB...
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2025-06-24 12:22:49,621] Trial 0 finished with value: 2.6402744486891083 and parameters: {'n_estimators': 1000, 'learning_rate': 0.010250582517387313, 'max_depth': 6, 'min_child_weight': 2, 'subsample': 0.6149501746489763, 'colsample_bytree': 0.612993571753984}. Best is trial 0 with value: 2.6402744486891083.


Running time: 2.4 sec
OOF RMSE: 2.6403
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2025-06-24 12:22:54,496] Trial 1 finished with value: 2.6551140502554027 and parameters: {'n_estimators': 1000, 'learning_rate': 0.006437532229785393, 'max_depth': 8, 'min_child_weight': 3, 'subsample': 0.9780500815920048, 'colsample_bytree': 0.7647705570506433}. Best is trial 0 with value: 2.6402744486891083.


...

Fold 5
Running time: 0.3 sec
OOF RMSE: 2.6784
Fold 1
Fold 2
Fold 3


[I 2025-06-24 12:24:09,456] Trial 18 finished with value: 2.7875795003296453 and parameters: {'learning_rate': 0.005596380240543389, 'num_leaves': 20, 'max_depth': 7, 'min_child_samples': 15, 'subsample': 0.9879610090915855, 'colsample_bytree': 0.8125020591488976, 'n_estimators': 1000}. Best is trial 11 with value: 2.6523885863923535.


Fold 4
Fold 5
Running time: 0.5 sec
OOF RMSE: 2.7876
Fold 1
Fold 2
Fold 3
Fold 4


[I 2025-06-24 12:24:09,798] Trial 19 finished with value: 2.6784534814223617 and parameters: {'learning_rate': 0.015045130475019163, 'num_leaves': 20, 'max_depth': 6, 'min_child_samples': 27, 'subsample': 0.8784043880020499, 'colsample_bytree': 0.855483063357705, 'n_estimators': 1000}. Best is trial 11 with value: 2.6523885863923535.


Fold 5
Running time: 0.3 sec
OOF RMSE: 2.6785

✅ LBM - Mejor RMSLE: 2.6524
📋 Parámetros: {'learning_rate': 0.08933589129532095, 'num_leaves': 20, 'max_depth': 7, 'min_child_samples': 27, 'subsample': 0.6011289390155303, 'colsample_bytree': 0.8961713183846663, 'n_estimators': 2000}



In [56]:
results

{'XGB': {'best_params': {'n_estimators': 2000,
   'learning_rate': 0.005038520390145823,
   'max_depth': 8,
   'min_child_weight': 1,
   'subsample': 0.8658250277664441,
   'colsample_bytree': 0.907105544104575},
  'best_score': 2.5861375238851174,
  'study': <optuna.study.study.Study at 0x705cdee95c70>},
 'LBM': {'best_params': {'learning_rate': 0.08933589129532095,
   'num_leaves': 20,
   'max_depth': 7,
   'min_child_samples': 27,
   'subsample': 0.6011289390155303,
   'colsample_bytree': 0.8961713183846663,
   'n_estimators': 2000},
  'best_score': 2.6523885863923535,
  'study': <optuna.study.study.Study at 0x705cdeddeb50>}}

In [30]:
#best_params = {'n_estimators': 2000, 'learning_rate': 0.04814113153747816, 'max_depth': 7, 'min_child_weight': 1, 'subsample': 0.9437728415677074, 'colsample_bytree': 0.909668290806239}
# {'n_estimators': 1000, 'learning_rate': 0.010767242978932143, 'max_depth': 7, 'min_child_weight': 1, 'subsample': 0.8684177755167737, 'colsample_bytree': 0.7303912217157997}
# {'n_estimators': 1000, 'learning_rate': 0.014410895948871282, 'max_depth': 8, 'min_child_weight': 1, 'subsample': 0.9990473761909979, 'colsample_bytree': 0.86335191588344}

#{'learning_rate': 0.08933589129532095, 'num_leaves': 20, 'max_depth': 7, 'min_child_samples': 27, 'subsample': 0.6011289390155303, 'colsample_bytree': 0.8961713183846663, 'n_estimators': 2000}
best_params_xgb = {'n_estimators': 1000, 'learning_rate': 0.012, 'max_depth': 7, 'min_child_weight': 1, 'subsample': 0.95, 'colsample_bytree': 0.85}
best_params_lbm = {'learning_rate': 0.089, 'num_leaves': 20, 'max_depth': 7, 'min_child_samples': 27, 'subsample': 0.60, 'colsample_bytree': 0.89, 'n_estimators': 2000}


In [18]:
params_xgb= {
    'n_estimators': 1000,
    'learning_rate': 0.012,
    'max_depth': 7,
    'min_child_weight': 1,
    'subsample': 0.95,
    'colsample_bytree': 0.85,
    'device': 'cpu',
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'enable_categorical': True,
    'early_stopping_rounds': 50,
    'eval_metric': 'rmse'}

params_lbm = {
    'learning_rate': 0.089,
    'num_leaves': 20,
    'max_depth': 7,
    'min_child_samples': 27,
    'subsample': 0.6,
    'colsample_bytree': 0.89,
    'n_estimators': 2000,
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'device': 'cpu',  
    'verbosity': -1,
    'early_stopping_rounds': 50}

### Entrenamiento XGB y LBM

In [41]:
save_folder = "training_results"
FOLDS = 5
kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

models = {
    'LightGBM': LGBMRegressor(**params_lbm),
    'XGBoost': XGBRegressor(**params_xgb)
}
models_storage = {name: [] for name in models}

results = {name: {'oof': np.zeros(len(train)), 'pred': np.zeros(len(test)), 'rmse': [], 'r2': [], 'r2_tfg' : []} for name in models}

target = "Chl"
X = train.drop(columns=[target, 'High_Chl'])
X_test = test.drop(columns=[target, 'High_Chl'])
y = train[target]
y_class = train["High_Chl"]

    
# Definimos los folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\n=== Training {name} ===")
    for i, (train_idx, valid_idx) in enumerate(skf.split(X, y_class)):
        print(f"\nFold {i+1}")
        x_train, y_train = X.iloc[train_idx], y[train_idx]
        x_valid, y_valid = X.iloc[valid_idx], y[valid_idx]
        
        start = time.time()
        
        if name == 'XGBoost':
            model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=500)
        elif name == 'LightGBM':
            model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)])

        models_storage[name].append(model)
        #joblib.dump(model, f"{name}_fold_{i}.pkl") 
        oof_pred = model.predict(x_valid)
        test_pred = model.predict(X_test)
        
        results[name]['oof'][valid_idx] = oof_pred
        results[name]['pred'] += test_pred / FOLDS
        
        rmse = np.sqrt(mean_squared_error(y_valid, oof_pred))
        r2 = r2_score(y_valid, oof_pred)
        corr = np.corrcoef(y_valid, oof_pred)[0, 1]
        r2_corr = corr ** 2
        results[name]['rmse'].append(rmse)
        results[name]['r2'].append(r2)
        results[name]['r2_tfg'].append(r2_corr)
        
        print(f"Fold {i+1} RMSE: {rmse:.2f} | R2: {r2:.2f} | R2_TFG {r2_corr:.2f}")
        print(f"Training time: {time.time() - start:.1f} sec")
    
    # np.save(f"{save_folder}/{name}_oof.npy", results[name]['oof'])
    # np.save(f"{save_folder}/{name}_pred.npy", results[name]['pred'])


print("\n=== Model Comparison ===")
for name in models:
    mean_rmse = np.mean(results[name]['rmse'])
    std_rmse = np.std(results[name]['rmse'])
    print(f"{name} - Mean RMSE: {mean_rmse:.2f} ± {std_rmse:.2f}")
    mean_r2 = np.mean(results[name]['r2'])
    std_r2 = np.std(results[name]['r2'])
    print(f"{name} - Mean R2: {mean_r2:.2f} ± {std_r2:.2f}")
    mean_r2_tfg = np.mean(results[name]['r2_tfg'])
    std_r2_tfg = np.std(results[name]['r2_tfg'])
    print(f"{name} - Mean R2: {mean_r2_tfg:.2f} ± {std_r2_tfg:.2f}")

    


=== Training LightGBM ===

Fold 1
Fold 1 RMSE: 2.99 | R2: 0.48 | R2_TFG 0.53
Training time: 0.0 sec

Fold 2
Fold 2 RMSE: 2.44 | R2: 0.35 | R2_TFG 0.37
Training time: 0.0 sec

Fold 3
Fold 3 RMSE: 2.89 | R2: 0.50 | R2_TFG 0.58
Training time: 0.0 sec

Fold 4
Fold 4 RMSE: 2.41 | R2: 0.57 | R2_TFG 0.58
Training time: 0.1 sec

Fold 5
Fold 5 RMSE: 2.58 | R2: 0.35 | R2_TFG 0.36
Training time: 0.0 sec

=== Training XGBoost ===

Fold 1
[0]	validation_0-rmse:4.14220
[427]	validation_0-rmse:3.18872
Fold 1 RMSE: 3.19 | R2: 0.41 | R2_TFG 0.42
Training time: 3.7 sec

Fold 2
[0]	validation_0-rmse:3.04737
[123]	validation_0-rmse:2.57395
Fold 2 RMSE: 2.53 | R2: 0.31 | R2_TFG 0.31
Training time: 1.3 sec

Fold 3
[0]	validation_0-rmse:4.08089
[500]	validation_0-rmse:2.66474
[999]	validation_0-rmse:2.65590
Fold 3 RMSE: 2.66 | R2: 0.58 | R2_TFG 0.65
Training time: 8.5 sec

Fold 4
[0]	validation_0-rmse:3.65318
[500]	validation_0-rmse:1.98663
[999]	validation_0-rmse:1.97681
Fold 4 RMSE: 1.98 | R2: 0.71 | R2_T

In [60]:
xgb_preds = np.load("training_results/XGBoost_pred.npy")
y_test = test[target]

rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
r2 = r2_score(y_test, xgb_preds)

print(f"XGB - Test RMSE: {rmse:.2f} | Test R2: {r2:.2f}")

lbm_preds = np.load("training_results/LightGBM_pred.npy")
y_test = test[target]

rmse = np.sqrt(mean_squared_error(y_test, lbm_preds))
r2 = r2_score(y_test, lbm_preds)

print(f"LBM - Test RMSE: {rmse:.2f} | Test R2: {r2:.2f}")

XGB - Test RMSE: 3.20 | Test R2: 0.54
LBM - Test RMSE: 3.48 | Test R2: 0.46


In [61]:
oof_xgb = np.load("training_results/XGBoost_oof.npy")
oof_lbm = np.load("training_results/LightGBM_oof.npy")
#oof_nn = np.load("NN_oof.npy")
pred_xgb = np.load("training_results/XGBoost_pred.npy")
pred_lbm = np.load("training_results/LightGBM_pred.npy")
#pred_nn = np.load("NN_pred.npy")

In [62]:
results = {
    'XGB': {'oof': oof_xgb, 'pred': pred_xgb},
    'LBM': {'oof': oof_lbm, 'pred': pred_lbm},   
    #'MLP': {'oof': oof_nn, 'pred': pred_nn},
}

In [64]:
oof_preds = {name: results[name]['oof'] for name in results}
test_preds = {name: results[name]['pred'] for name in results}
y_true = y

In [65]:
meta_X = np.vstack([oof_preds["XGB"], oof_preds["LBM"]]).T
meta_y = y_true  # target original

# Luego entrenar un meta-modelo
from sklearn.linear_model import Ridge
meta_model = Ridge().fit(meta_X, meta_y)
rmse = np.sqrt(mean_squared_error(meta_y, meta_model.predict(meta_X)))
r2 = r2_score(meta_y, meta_model.predict(meta_X))
print(f"RMSE: {rmse:.2f} | R2: {r2:.2f}")

RMSE: 2.55 | R2: 0.52


In [66]:
meta_X_test = np.vstack([test_preds["XGB"], test_preds["LBM"]]).T
test_pred = meta_model.predict(meta_X_test)
rmse = np.sqrt(mean_squared_error(y_test, test_pred))
r2 = r2_score(y_test, test_pred)
print(f"RMSE: {rmse:.2f} | R2: {r2:.2f}")

RMSE: 3.22 | R2: 0.54


### TabPFN

In [70]:
from tabpfn import TabPFNRegressor  

ImportError: cannot import name 'TabPFNRegressor' from 'tabpfn' (/home/antonio/.pyenv/versions/3.8.5/envs/nitrates/lib/python3.8/site-packages/tabpfn/__init__.py)

In [69]:
X_train = train.drop(columns=[target, 'High_Chl'])
X_test = test.drop(columns=[target, 'High_Chl'])
y = train[target]
y_test = test[target]

,rhow_B1,rhow_B2,rhow_B4,rhow_B5,Band_15,dif_norm_rhow_B2_B3,dif_inv_rhow_B2_B3,dif_norm_rhow_B2_B4,dif_inv_rhow_B2_B4,dif_norm_rhow_B3_B4,...,dif_rel_4bands_rhow_B3_B4_B5_B2,sum_norm_3bands_rhow_B2_B4_B3,sum_norm_3bands_rhow_B3_B5_B4,dall_gitelson_rhown_B3_B4_B5,dif_norm_4_bands_rhown_B4_B5_B2_B3,sum_norm_3bands_rhown_B3_B5_B4,Chl,High_Chl,Season,Zone
0,0.020469,0.031289,0.011193,0.007345,0.147099,-0.079,4.681,0.473,-57.385,0.532,...,3.040,0.625,0.920,-0.466,0.054,0.923,0.4995,False,Verano,Zona-4
1,0.009607,0.021231,0.010398,0.007248,0.115210,-0.313,22.437,0.342,-49.068,0.592,...,3.558,0.512,0.938,-0.527,0.049,0.940,1.6950,False,Primavera,Zona-3
2,0.009486,0.020363,0.015713,0.011596,0.166120,-0.352,25.583,0.129,-14.534,0.460,...,2.136,0.574,0.929,-0.478,0.061,0.934,2.9870,False,Primavera,Zona-3
3,0.004488,0.011016,0.008279,0.006037,0.088587,-0.372,49.260,0.142,-30.012,0.488,...,2.361,0.550,0.931,-0.483,0.062,0.932,2.4695,False,Primavera,Zona-1
4,0.001168,0.003506,0.001086,0.000560,0.000027,-0.323,139.403,0.527,-635.927,0.727,...,6.156,0.443,0.934,-0.484,0.042,0.944,0.6580,False,Otoño,Zona-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,0.006617,0.012153,0.002676,0.001761,0.035969,-0.001,0.211,0.639,-291.347,0.640,...,4.408,0.609,0.938,-0.556,0.031,0.948,0.5110,False,Primavera,Zona-2
332,0.010938,0.015758,0.009683,0.007627,0.216871,-0.166,18.093,0.239,-39.821,0.390,...,1.793,0.673,0.935,-0.445,0.053,0.937,0.4490,False,Otoño,Zona-2
333,0.001308,0.004138,0.001208,0.000618,0.000027,-0.327,119.011,0.548,-585.858,0.742,...,6.597,0.435,0.937,-0.488,0.039,0.948,0.3030,False,Primavera,Zona-1
334,0.003544,0.010456,0.027829,0.024688,0.332638,-0.555,68.239,-0.454,59.704,0.135,...,-1.050,0.815,0.951,-0.257,0.060,0.955,4.3660,False,Otoño,Zona-2


In [ ]:
# Initialize the regressor
regressor = TabPFNRegressor()  
regressor.fit(X_train, y_train)

# Predict on the test set
predictions = regressor.predict(X_test)

# Evaluate the model

rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("RMSE):", rmse)
print("R² Score:", r2)